# Moon Valley Nursery ML Models

This Snowflake Notebook trains, evaluates, and **registers** four machine learning models in the Snowflake Model Registry (`MVN_INTELLIGENCE.ANALYTICS`):

1. **`WEEKLY_SALES_FORECAST_MODEL` (Regression)** — Seasonality-aware store × category weekly sales forecast
2. **`STOCKOUT_RISK_MODEL` (Binary Classification)** — Predicts 14-day stockout risk by store and product SKU
3. **`PLANTING_ATTACH_MODEL` (Binary Classification)** — Predicts transaction-level professional planting attach propensity
4. **`PLANTING_DELAY_RISK_MODEL` (Binary Classification)** — Predicts whether a scheduled planting job is at risk of delay/reschedule

**CRITICAL**: All models are registered via `Registry.log_model()`, logged with evaluation metrics via `mv.set_metric()`, and called by SQL UDFs via `MODEL()!PREDICT()`. No hardcoded thresholds are used.

In [ ]:
# Cell 1: Session setup — ONLY way to get session in Snowflake Notebooks
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f'Connected: {session.get_current_database()}.{session.get_current_schema()}')

In [ ]:
-- result_variable_name: df_sales
-- Load weekly store x category sales features for Weekly Sales Forecast Model
SELECT
    STATE_CODE,
    CATEGORY_CODE,
    SEASON_CODE,
    IS_PEAK_SEASON_NUM,
    SQ_FT,
    STORE_AGE_MONTHS,
    PRIOR_4WK_AVG_SALES,
    WEEKLY_NET_SALES
FROM MVN_INTELLIGENCE.ANALYTICS.V_WEEKLY_SALES_STORE_CATEGORY

In [ ]:
-- result_variable_name: df_stockout
-- Load daily store x product inventory features for 14-Day Stockout Risk Classifier
SELECT
    ON_HAND_UNITS,
    ON_ORDER_UNITS,
    DAILY_SELL_RATE,
    DAYS_OF_SUPPLY,
    LEAD_TIME_DAYS,
    IS_PEAK_SEASON_NUM,
    IS_PALM_NUM,
    STOCKOUT_NEXT_14D_FLAG
FROM MVN_INTELLIGENCE.ANALYTICS.V_ML_STOCKOUT_FEATURES

In [ ]:
-- result_variable_name: df_attach
-- Load transaction-level features for Planting Attach Propensity Classifier
SELECT
    CUSTOMER_TYPE_CODE,
    TREE_UNITS,
    BOX_TREE_UNITS,
    TOTAL_UNITS,
    GROSS_SALES,
    IS_PEAK_SEASON_NUM,
    INCLUDES_PLANTING_NUM
FROM MVN_INTELLIGENCE.ANALYTICS.V_ML_PLANTING_ATTACH_FEATURES

In [ ]:
-- result_variable_name: df_delay
-- Load planting job features for Planting Delay Risk Classifier
SELECT
    DAYS_TO_SCHEDULE,
    TREE_COUNT,
    MAX_CONTAINER_CODE,
    CREW_SIZE,
    IS_PEAK_SEASON_NUM,
    CRANE_CERTIFIED_NUM,
    DELAYED_FLAG
FROM MVN_INTELLIGENCE.ANALYTICS.V_ML_PLANTING_DELAY_FEATURES

In [ ]:
# Cell 6: Import libraries and verify SQL cell results
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score

print(f'Weekly sales rows: {len(df_sales)}')
print(f'Stockout risk rows: {len(df_stockout)}')
print(f'Planting attach rows: {len(df_attach)}')
print(f'Planting delay rows: {len(df_delay)}')

In [ ]:
# Cell 7: MODEL 1 — WEEKLY_SALES_FORECAST_MODEL (Regression)
sales_features = [
    'STATE_CODE',
    'CATEGORY_CODE',
    'SEASON_CODE',
    'IS_PEAK_SEASON_NUM',
    'SQ_FT',
    'STORE_AGE_MONTHS',
    'PRIOR_4WK_AVG_SALES'
]
sales_target = 'WEEKLY_NET_SALES'

df_sales_clean = df_sales[sales_features + [sales_target]].dropna()
print(f'Weekly Sales Forecast training rows: {len(df_sales_clean)}')

X_sales = df_sales_clean[sales_features]
y_sales = df_sales_clean[sales_target]

X_train_sales, X_test_sales, y_train_sales, y_test_sales = train_test_split(
    X_sales, y_sales, test_size=0.2, random_state=42
)

sales_model = GradientBoostingRegressor(
    n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42
)
sales_model.fit(X_train_sales, y_train_sales)

y_pred_sales = sales_model.predict(X_test_sales)
sales_mae = mean_absolute_error(y_test_sales, y_pred_sales)
sales_r2 = r2_score(y_test_sales, y_pred_sales)
print('Weekly Sales Forecast Model Performance:')
print(f'  MAE: ${sales_mae:,.2f}')
print(f'  R2 Score: {sales_r2:.4f}')

In [ ]:
# Cell 8: MODEL 2 — STOCKOUT_RISK_MODEL (Binary Classification)
stockout_features = [
    'ON_HAND_UNITS',
    'ON_ORDER_UNITS',
    'DAILY_SELL_RATE',
    'DAYS_OF_SUPPLY',
    'LEAD_TIME_DAYS',
    'IS_PEAK_SEASON_NUM',
    'IS_PALM_NUM'
]
stockout_target = 'STOCKOUT_NEXT_14D_FLAG'

df_so_clean = df_stockout[stockout_features + [stockout_target]].dropna()
print(f'Stockout Risk training rows: {len(df_so_clean)}')

X_so = df_so_clean[stockout_features]
y_so = df_so_clean[stockout_target]

X_train_so, X_test_so, y_train_so, y_test_so = train_test_split(
    X_so, y_so, test_size=0.2, random_state=42, stratify=y_so
)

stockout_model = GradientBoostingClassifier(
    n_estimators=120, max_depth=4, learning_rate=0.1, random_state=42
)
stockout_model.fit(X_train_so, y_train_so)

y_pred_so = stockout_model.predict(X_test_so)
so_accuracy = accuracy_score(y_test_so, y_pred_so)
so_f1 = f1_score(y_test_so, y_pred_so)
print('Stockout Risk Model Performance:')
print(f'  Accuracy: {so_accuracy:.4f}')
print(f'  F1 Score: {so_f1:.4f}')

In [ ]:
# Cell 9: MODEL 3 — PLANTING_ATTACH_MODEL (Binary Classification)
attach_features = [
    'CUSTOMER_TYPE_CODE',
    'TREE_UNITS',
    'BOX_TREE_UNITS',
    'TOTAL_UNITS',
    'GROSS_SALES',
    'IS_PEAK_SEASON_NUM'
]
attach_target = 'INCLUDES_PLANTING_NUM'

df_att_clean = df_attach[attach_features + [attach_target]].dropna()
print(f'Planting Attach training rows: {len(df_att_clean)}')

X_att = df_att_clean[attach_features]
y_att = df_att_clean[attach_target]

X_train_att, X_test_att, y_train_att, y_test_att = train_test_split(
    X_att, y_att, test_size=0.2, random_state=42, stratify=y_att
)

attach_model = GradientBoostingClassifier(
    n_estimators=120, max_depth=4, learning_rate=0.1, random_state=42
)
attach_model.fit(X_train_att, y_train_att)

y_pred_att = attach_model.predict(X_test_att)
att_accuracy = accuracy_score(y_test_att, y_pred_att)
att_f1 = f1_score(y_test_att, y_pred_att)
print('Planting Attach Model Performance:')
print(f'  Accuracy: {att_accuracy:.4f}')
print(f'  F1 Score: {att_f1:.4f}')

In [ ]:
# Cell 10: MODEL 4 — PLANTING_DELAY_RISK_MODEL (Binary Classification)
delay_features = [
    'DAYS_TO_SCHEDULE',
    'TREE_COUNT',
    'MAX_CONTAINER_CODE',
    'CREW_SIZE',
    'IS_PEAK_SEASON_NUM',
    'CRANE_CERTIFIED_NUM'
]
delay_target = 'DELAYED_FLAG'

df_del_clean = df_delay[delay_features + [delay_target]].dropna()
print(f'Planting Delay Risk training rows: {len(df_del_clean)}')

X_del = df_del_clean[delay_features]
y_del = df_del_clean[delay_target]

X_train_del, X_test_del, y_train_del, y_test_del = train_test_split(
    X_del, y_del, test_size=0.2, random_state=42, stratify=y_del
)

delay_model = GradientBoostingClassifier(
    n_estimators=120, max_depth=4, learning_rate=0.1, random_state=42
)
delay_model.fit(X_train_del, y_train_del)

y_pred_del = delay_model.predict(X_test_del)
del_accuracy = accuracy_score(y_test_del, y_pred_del)
del_f1 = f1_score(y_test_del, y_pred_del)
print('Planting Delay Risk Model Performance:')
print(f'  Accuracy: {del_accuracy:.4f}')
print(f'  F1 Score: {del_f1:.4f}')

In [ ]:
# Cell 11: REGISTER ALL FOUR MODELS IN SNOWFLAKE MODEL REGISTRY
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name='MVN_INTELLIGENCE', schema_name='ANALYTICS')

# --- 1. Register Weekly Sales Forecast Model ---
print('Registering WEEKLY_SALES_FORECAST_MODEL...')
mv_sales = registry.log_model(
    sales_model,
    model_name='WEEKLY_SALES_FORECAST_MODEL',
    version_name='V1',
    sample_input_data=X_train_sales.head(50),
    conda_dependencies=['scikit-learn'],
    comment='Regression: forecasts weekly net sales by store and product category. Features: state_code, category_code, season_code, is_peak_season_num, sq_ft, store_age_months, prior_4wk_avg_sales.'
)
mv_sales.set_metric('mae', float(sales_mae))
mv_sales.set_metric('r2_score', float(sales_r2))
print(f'  Registered WEEKLY_SALES_FORECAST_MODEL/V1 (MAE=${sales_mae:,.2f}, R2={sales_r2:.4f})')

# --- 2. Register Stockout Risk Model ---
print('\nRegistering STOCKOUT_RISK_MODEL...')
mv_so = registry.log_model(
    stockout_model,
    model_name='STOCKOUT_RISK_MODEL',
    version_name='V1',
    sample_input_data=X_train_so.head(50),
    conda_dependencies=['scikit-learn'],
    comment='Binary classification: predicts 14-day stockout risk by store and product. Features: on_hand_units, on_order_units, daily_sell_rate, days_of_supply, lead_time_days, is_peak_season_num, is_palm_num.'
)
mv_so.set_metric('accuracy', float(so_accuracy))
mv_so.set_metric('f1_score', float(so_f1))
print(f'  Registered STOCKOUT_RISK_MODEL/V1 (Accuracy={so_accuracy:.4f}, F1={so_f1:.4f})')

# --- 3. Register Planting Attach Propensity Model ---
print('\nRegistering PLANTING_ATTACH_MODEL...')
mv_att = registry.log_model(
    attach_model,
    model_name='PLANTING_ATTACH_MODEL',
    version_name='V1',
    sample_input_data=X_train_att.head(50),
    conda_dependencies=['scikit-learn'],
    comment='Binary classification: predicts transaction-level professional planting attach propensity. Features: customer_type_code, tree_units, box_tree_units, total_units, gross_sales, is_peak_season_num.'
)
mv_att.set_metric('accuracy', float(att_accuracy))
mv_att.set_metric('f1_score', float(att_f1))
print(f'  Registered PLANTING_ATTACH_MODEL/V1 (Accuracy={att_accuracy:.4f}, F1={att_f1:.4f})')

# --- 4. Register Planting Delay Risk Model ---
print('\nRegistering PLANTING_DELAY_RISK_MODEL...')
mv_del = registry.log_model(
    delay_model,
    model_name='PLANTING_DELAY_RISK_MODEL',
    version_name='V1',
    sample_input_data=X_train_del.head(50),
    conda_dependencies=['scikit-learn'],
    comment='Binary classification: predicts whether a scheduled planting job will miss its scheduled date or require rescheduling. Features: days_to_schedule, tree_count, max_container_code, crew_size, is_peak_season_num, crane_certified_num.'
)
mv_del.set_metric('accuracy', float(del_accuracy))
mv_del.set_metric('f1_score', float(del_f1))
print(f'  Registered PLANTING_DELAY_RISK_MODEL/V1 (Accuracy={del_accuracy:.4f}, F1={del_f1:.4f})')

print('\n--- ALL 4 MODELS REGISTERED SUCCESSFULLY ---')

In [ ]:
# Cell 12: VERIFY MODEL REGISTRY AND TEST INFERENCE
print('Models registered in MVN_INTELLIGENCE.ANALYTICS:')
models_df = registry.show_models()
print(models_df[['name', 'versions', 'comment']].to_string())

print('\n--- Testing inference via Model Registry ---')
res_sales = mv_sales.run(X_test_sales.head(5))
print(f'Weekly Sales Forecast (5 samples): {[round(x, 2) for x in res_sales["output_feature_0"].tolist()]}')

res_so = mv_so.run(X_test_so.head(5), function_name='predict')
print(f'Stockout Risk (5 samples): {res_so["output_feature_0"].tolist()}')

res_att = mv_att.run(X_test_att.head(5), function_name='predict')
print(f'Planting Attach Propensity (5 samples): {res_att["output_feature_0"].tolist()}')

res_del = mv_del.run(X_test_del.head(5), function_name='predict')
print(f'Planting Delay Risk (5 samples): {res_del["output_feature_0"].tolist()}')

print('\n=== PIPELINE VERIFIED: train -> register -> predict ===')